# Contract Pipeline Test Report

Notebook này tổng hợp kết quả test cho các file hợp đồng trong `data/output` và so sánh với ground truth trong `data/ground_truth`.

Các metric chính:
- `expected`: số keyword/provision trong ground truth
- `found`: số keyword/provision pipeline tìm được
- `matched`: số representative keyword match exact sau normalize lowercase
- `precision`: matched / found
- `recall`: matched / expected
- `evidence_passed`: số evidence có `validation_status = passed`
- `exact_text_found`: số evidence có `exact_text` tìm thấy trong document text hoặc segment text


## 1. Ground-Truth Template

**Finding:** the ground-truth file should be a scoring template for contract extraction. It must include the uploaded file name, the expected representative keyword/provision, optional related keywords, and expected metadata so results can be matched and scored consistently.

**Current supported JSON shape:**

```json
{
  "document_name": "testoutput.pdf",
  "source_file": "data/sample_docs/testoutput.pdf",
  "keyword_groups": [
    {
      "representative_keyword": "Payment Terms",
      "provision_type": "payment",
      "related_keywords": ["fees", "invoice"],
      "metadata": {"page": 2, "source": "data/sample_docs/testoutput.pdf"}
    }
  ]
}
```

**Corrected template (recommended):**

| Column | Description | Required |
|---|---|---|
| `document_name` | Exact uploaded document filename, e.g. `testoutput.pdf` | Yes |
| `source_file` | Local path to the source document, e.g. `data/sample_docs/testoutput.pdf` | Recommended |
| `representative_keyword` | Canonical keyword/provision expected from LLM1 | Yes |
| `provision_type` | One of the pipeline provision types, e.g. `payment`, `termination`, `confidentiality` | Recommended |
| `related_keywords` | Phrases that may appear under the same representative keyword | Recommended |
| `metadata.page` | Page where the provision appears | Recommended |
| `metadata.source` | Source document path or filename | Recommended |
| `evidences[].exact_text` | Optional exact evidence text expected from LLM2 | Optional |

**Semantics:** for this file, this representative keyword/provision should be present in the extraction result. One row/object per `(document x provision)`; the template should be filled for every uploaded document that needs scoring.

**Validation rules:** required fields present; at least one keyword group; skip or report rows missing `representative_keyword`; compare result and ground truth by normalized `representative_keyword`; report `missing` and `extra`; optionally validate LLM2 `exact_text` against the source document. Supported format in this notebook: JSON.


In [1]:
from __future__ import annotations

import json
import re
import sys
from pathlib import Path
from typing import Any

ROOT = Path.cwd()
if not (ROOT / 'app').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

RESULT_DIR = ROOT / 'data' / 'output'
GROUND_TRUTH_DIR = ROOT / 'data' / 'ground_truth'
SAMPLE_DOC_DIRS = [ROOT / 'data' / 'sample_docs', ROOT / 'data' / 'input']
REPORT_PATH = ROOT / 'data' / 'output' / 'contract_test_report.md'

print('ROOT =', ROOT)
print('RESULT_DIR =', RESULT_DIR)
print('GROUND_TRUTH_DIR =', GROUND_TRUTH_DIR)


ROOT = d:\keyword_pipeline
RESULT_DIR = d:\keyword_pipeline\data\output
GROUND_TRUTH_DIR = d:\keyword_pipeline\data\ground_truth


In [2]:
def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding='utf-8'))


def norm_keyword(value: Any) -> str:
    text = str(value or '').strip().lower()
    text = re.sub(r'[_\u2010-\u2015-]+', ' ', text)
    text = re.sub(r'[^a-z0-9]+', ' ', text)
    return ' '.join(text.split())


def percent(numerator: int | float, denominator: int | float) -> float:
    return float(numerator) / float(denominator) if denominator else 0.0


def doc_key(document_name: str | None) -> str:
    name = (document_name or '').strip().lower()
    name = re.sub(r'\(\d+\)(?=\.[^.]+$)', '', name)
    return name


def representative_keywords(data: dict[str, Any]) -> dict[str, str]:
    output = {}
    for group in data.get('keyword_groups', []):
        keyword = group.get('representative_keyword')
        normalized = norm_keyword(keyword)
        if normalized:
            output[normalized] = str(keyword)
    return output


def find_source_path(document_name: str | None) -> Path | None:
    if not document_name:
        return None
    for directory in SAMPLE_DOC_DIRS:
        candidate = directory / document_name
        if candidate.exists():
            return candidate
    return None


def extract_document_text(path: Path | None) -> str:
    if path is None or not path.exists():
        return ''
    if path.suffix.lower() == '.txt':
        return path.read_text(encoding='utf-8', errors='ignore')
    if path.suffix.lower() == '.pdf':
        try:
            from pypdf import PdfReader
            reader = PdfReader(str(path))
            return '\n'.join(page.extract_text() or '' for page in reader.pages)
        except Exception as error:
            print('PDF extract failed:', path, error)
    return ''


def evidence_stats(result: dict[str, Any]) -> dict[str, int | float]:
    source_path = find_source_path(result.get('document_name'))
    document_text = extract_document_text(source_path)
    total = passed = not_found = exact_text_found = exact_text_missing = exact_text_total = 0
    for group in result.get('keyword_groups', []):
        for evidence in group.get('evidences', []):
            total += 1
            status = evidence.get('validation_status')
            if status == 'passed':
                passed += 1
            if status == 'not_found':
                not_found += 1
            exact_text = evidence.get('exact_text') or ''
            if exact_text:
                exact_text_total += 1
                if exact_text in document_text:
                    exact_text_found += 1
                else:
                    exact_text_missing += 1
    return {
        'evidence_total': total,
        'evidence_passed': passed,
        'evidence_not_found': not_found,
        'exact_text_total': exact_text_total,
        'exact_text_found': exact_text_found,
        'exact_text_missing': exact_text_missing,
        'evidence_exact_accuracy': percent(exact_text_found, exact_text_total),
    }


def compare_result_to_truth(result_path: Path, truth_path: Path) -> dict[str, Any]:
    result = load_json(result_path)
    truth = load_json(truth_path)
    found = representative_keywords(result)
    expected = representative_keywords(truth)
    matched_keys = set(found) & set(expected)
    missing_keys = set(expected) - set(found)
    extra_keys = set(found) - set(expected)
    found_count = len(found)
    expected_count = len(expected)
    matched_count = len(matched_keys)
    stats = evidence_stats(result)
    return {
        'result_file': result_path.name,
        'ground_truth_file': truth_path.name,
        'document_name': result.get('document_name'),
        'processing_time_seconds': result.get('processing_time_seconds', 0),
        'total_pages': result.get('total_pages', 0),
        'total_segments': result.get('total_segments', 0),
        'expected': expected_count,
        'found': found_count,
        'matched': matched_count,
        'extract_accuracy_vs_ground_truth': percent(matched_count, expected_count),
        'extract_precision': percent(matched_count, found_count),
        'missing': [expected[key] for key in sorted(missing_keys)],
        'extra': [found[key] for key in sorted(extra_keys)],
        **stats,
    }


In [3]:
result_files = sorted(RESULT_DIR.glob('*.json'))
truth_files = sorted(GROUND_TRUTH_DIR.glob('*.json'))

truth_by_doc: dict[str, list[Path]] = {}
for truth_path in truth_files:
    truth = load_json(truth_path)
    truth_by_doc.setdefault(doc_key(truth.get('document_name')), []).append(truth_path)

pairs: list[tuple[Path, Path]] = []
unmatched_results = []
for result_path in result_files:
    result = load_json(result_path)
    key = doc_key(result.get('document_name'))
    matches = truth_by_doc.get(key, [])
    if matches:
        for truth_path in matches:
            pairs.append((result_path, truth_path))
    else:
        unmatched_results.append(result_path)

rows = [compare_result_to_truth(result_path, truth_path) for result_path, truth_path in pairs]
rows = sorted(rows, key=lambda row: (row['document_name'] or '', row['result_file'], row['ground_truth_file']))

print('Result files:', len(result_files))
print('Ground truth files:', len(truth_files))
print('Matched result/ground-truth pairs:', len(rows))
print('Unmatched result files:', [path.name for path in unmatched_results])


Result files: 5
Ground truth files: 6
Matched result/ground-truth pairs: 0
Unmatched result files: ['1.json', '1_original.json', '2.json', '4.json', '6.json']


In [4]:
from IPython.display import Markdown, display


def fmt_seconds(value: Any) -> str:
    try:
        seconds = float(value)
    except (TypeError, ValueError):
        return '0.00s'
    if seconds >= 60:
        return f'{seconds / 60:.2f}m'
    return f'{seconds:.2f}s'


def markdown_table(rows: list[dict[str, Any]]) -> str:
    headers = [
        'document_name',
        'result_file',
        'ground_truth_file',
        'processing_time',
        'expected',
        'found',
        'matched',
        'extract_accuracy_vs_ground_truth',
        'extract_precision',
        'evidence_exact_accuracy',
        'exact_text_missing',
    ]
    labels = {
        'document_name': 'document',
        'result_file': 'result',
        'ground_truth_file': 'ground truth',
        'processing_time': 'time',
        'expected': 'GT expected',
        'found': 'extracted',
        'matched': 'matched',
        'extract_accuracy_vs_ground_truth': 'accuracy vs GT',
        'extract_precision': 'precision',
        'evidence_exact_accuracy': 'evidence exact accuracy',
        'exact_text_missing': 'exact text missing',
    }
    lines = ['|' + '|'.join(labels[h] for h in headers) + '|', '|' + '|'.join(['---'] * len(headers)) + '|']
    for row in rows:
        values = []
        for header in headers:
            if header == 'processing_time':
                value = fmt_seconds(row.get('processing_time_seconds'))
            else:
                value = row.get(header, '')
            if isinstance(value, float):
                value = f'{value:.2%}'
            values.append(str(value).replace('|', '\\|'))
        lines.append('|' + '|'.join(values) + '|')
    return '\n'.join(lines)


summary_lines = ['## 2. Test Summary', '']
if rows:
    avg_time = sum(float(row.get('processing_time_seconds') or 0) for row in rows) / len(rows)
    avg_accuracy = sum(row['extract_accuracy_vs_ground_truth'] for row in rows) / len(rows)
    avg_precision = sum(row['extract_precision'] for row in rows) / len(rows)
    summary_lines.append(
        f"**Finding:** evaluated `{len(rows)}` result/ground-truth pair(s). "
        f"Average processing time `{fmt_seconds(avg_time)}`, "
        f"average accuracy vs ground truth `{avg_accuracy:.2%}`, "
        f"average extraction precision `{avg_precision:.2%}`."
    )
    summary_lines.append('')
    summary_lines.append(markdown_table(rows))
else:
    summary_lines.append(
        '**Finding:** no result/ground-truth pairs were found. '
        'Place result JSON files in `data/output` and ground-truth JSON files in `data/ground_truth`, '
        'with matching `document_name`, then rerun the notebook.'
    )
summary_md = '\n'.join(summary_lines)
display(Markdown(summary_md))


## 2. Test Summary

**Finding:** no result/ground-truth pairs were found. Place result JSON files in `data/output` and ground-truth JSON files in `data/ground_truth`, with matching `document_name`, then rerun the notebook.

In [5]:
detail_lines = ['## 3. Result Findings']
for row in rows:
    detail_lines.append(f"\n### {row['result_file']} vs {row['ground_truth_file']}")
    accuracy = row['extract_accuracy_vs_ground_truth']
    precision = row['extract_precision']
    finding = 'coverage gap' if accuracy < 0.8 else 'coverage acceptable'
    if row['exact_text_missing']:
        finding += ' + evidence exact-text issue'
    detail_lines.append(
        f"**Finding:** `{finding}` - processing time `{fmt_seconds(row['processing_time_seconds'])}`; "
        f"matched `{row['matched']}` of `{row['expected']}` ground-truth provisions."
    )
    detail_lines.append(
        f"**Extraction score:** accuracy vs ground truth `{accuracy:.2%}`; "
        f"precision `{precision:.2%}` based on `{row['found']}` extracted provisions."
    )
    detail_lines.append(
        f"**Evidence score:** exact-text accuracy `{row['evidence_exact_accuracy']:.2%}` "
        f"(`{row['exact_text_found']}/{row['exact_text_total']}` exact_text values found in source); "
        f"passed `{row['evidence_passed']}`, not_found `{row['evidence_not_found']}`."
    )
    if row['missing']:
        detail_lines.append('**Missing expected provisions:** ' + ', '.join(f'`{item}`' for item in row['missing'][:30]))
    if row['extra']:
        detail_lines.append('**Extra extracted provisions:** ' + ', '.join(f'`{item}`' for item in row['extra'][:30]))
    detail_lines.append('**Recommended check:** review missing/extra pairs first for synonym naming differences, then inspect any `exact_text_missing` rows in LLM2 evidence output.')

details_md = '\n'.join(detail_lines)
display(Markdown(details_md))


## 3. Result Findings

In [6]:
report_md = '# Contract Pipeline Test Report\n\n' + summary_md + '\n\n' + details_md + '\n'
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(report_md, encoding='utf-8')
print('Wrote report to:', REPORT_PATH)


Wrote report to: d:\keyword_pipeline\data\output\contract_test_report.md


## Notes

- **Accuracy vs ground truth** = `matched / expected`. This answers: "so v?i ground truth, pipeline extract ??ng bao nhi?u ph?n tr?m n?i dung/provision mong ??i?"
- **Precision** = `matched / found`. This answers: "trong nh?ng provision pipeline extract ra, bao nhi?u ph?n tr?m c? trong ground truth?"
- Matching hi?n t?i l? exact match sau normalize `representative_keyword`. C?c keyword g?n ngh?a nh? `Development Fees` vs `Development Fee` c? th? b? t?nh l? `extra/missing`.
- **Evidence exact accuracy** checks whether LLM2 `exact_text` appears in the source PDF/TXT text. This is a quick sanity check, not a full segment-level validator.
- N?u mu?n report semantic match, b??c ti?p theo n?n th?m synonym map ho?c embedding/fuzzy matching cho representative keywords.
